# Import Functions

In [0]:
from pyspark.sql.functions import *

# Establish the Connection with ADLS

In [0]:
# Azure Storage authentication
# Configure Azure Storage credentials securely before execution.
# Do not store access keys or secrets in this notebook.

# Load all Dimensions

### Customer Dimensions

In [0]:
df_customer = spark.sql(
"""
select * from delta.`abfss://gold@cfstorageaccountrkd.dfs.core.windows.net/dim_customer`
"""
)

df_customer.display()

In [0]:
df_count = spark.sql(
"""
SELECT COUNT(*) AS total_rows
FROM delta.`abfss://gold@cfstorageaccountrkd.dfs.core.windows.net/dim_customer`
"""
)

df_count.display()

### Product Dimension

In [0]:
df_product = spark.sql(
"""
select * from delta.`abfss://gold@cfstorageaccountrkd.dfs.core.windows.net/dim_product`
"""
)

df_product.display()

### Date Dimension

In [0]:
df_date = spark.sql(
"""
select * from delta.`abfss://gold@cfstorageaccountrkd.dfs.core.windows.net/dim_date`
"""
)

df_date.display()

### Region Dimension

In [0]:
df_region = spark.sql(
"""
select * from delta.`abfss://gold@cfstorageaccountrkd.dfs.core.windows.net/dim_region`
"""
)

df_region.display()

### Load Silver Data

In [0]:
silver_df = spark.sql(
""" 
select * from parquet.`abfss://silver@cfstorageaccountrkd.dfs.core.windows.net/silverdata`
"""

)

silver_df.display()

### Create a Fact Table

In [0]:

fact_df = (
    silver_df
    .join(
        df_customer,
        silver_df["CustomerID"] == df_customer["CustomerId"],
        "left"
    )
    .join(
        df_product,
        silver_df["ProductID"] == df_product["ProductID"],
        "left"
    )
    .join(
        df_date,
        silver_df["OrderDate"] == df_date["OrderDate"],
        "left"
    )
    .join(
        df_region,
        silver_df["SalesRegion"] == df_region["SalesRegion"],
        "left"
    )
    .select(
        df_customer["Dim_Customer_Key"],
        df_product["Dim_Product_Key"],
        df_date["Dim_Date_Key"],
        df_region["Dim_Region_Key"],
        silver_df["UnitPrice"],
        silver_df["TotalPrice"],
        silver_df["Quantity"]
    )
)

fact_df.display()




#SCD Type 1 - Upsert
- SCD (Slowly Changing Dimenstion)
- ACID Transaction - Atomicity Consistency Isolation Durability

In [0]:

from delta.tables import DeltaTable

table_name = "gold.dim_factsales"

path = "abfss://gold@cfstorageaccountrkd.dfs.core.windows.net/dim_factsales"

# Read source data from Silver layer
df = spark.sql("""
    SELECT
        OrderID,
        OrderDate,
        CustomerID,
        CustomerName,
        Country,
        ProductID,
        ProductCategory,
        ProductName,
        Quantity,
        UnitPrice,
        TotalPrice,
        SalesRegion
    FROM parquet.`abfss://silver@cfstorageaccountrkd.dfs.core.windows.net/silverdata`
""")

if DeltaTable.isDeltaTable(spark, path):

    deltaTable = DeltaTable.forPath(spark, path)

    deltaTable.alias("target").merge(
        df.alias("source"),
        """
        target.OrderID = source.OrderID
        AND target.ProductID = source.ProductID
        """
    ).whenMatchedUpdate(set={
        "OrderDate": "source.OrderDate",
        "CustomerID": "source.CustomerID",
        "CustomerName": "source.CustomerName",
        "Country": "source.Country",
        "ProductCategory": "source.ProductCategory",
        "ProductName": "source.ProductName",
        "Quantity": "source.Quantity",
        "UnitPrice": "source.UnitPrice",
        "TotalPrice": "source.TotalPrice",
        "SalesRegion": "source.SalesRegion"
    }).whenNotMatchedInsert(values={
        "OrderID": "source.OrderID",
        "OrderDate": "source.OrderDate",
        "CustomerID": "source.CustomerID",
        "CustomerName": "source.CustomerName",
        "Country": "source.Country",
        "ProductID": "source.ProductID",
        "ProductCategory": "source.ProductCategory",
        "ProductName": "source.ProductName",
        "Quantity": "source.Quantity",
        "UnitPrice": "source.UnitPrice",
        "TotalPrice": "source.TotalPrice",
        "SalesRegion": "source.SalesRegion"
    }).execute()

    print("Good! Data merged successfully")

else:

    df.write.format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .save(path)

    print("The table is created")


# Testing

In [0]:
df = spark.sql("""
    SELECT *
    FROM parquet.`abfss://silver@cfstorageaccountrkd.dfs.core.windows.net/silverdata`
""")

print("Total Source Records:", df.count())

In [0]:
files = dbutils.fs.ls(
    "abfss://silver@cfstorageaccountrkd.dfs.core.windows.net/silverdata/"
)

for f in files:
    print(f.name, f.size)

In [0]:
file_path = [f.path for f in files if f.name.endswith(".parquet")][0]

df_check = spark.read.parquet(file_path)

print("Total Records:", df_check.count())

display(df_check)